In [0]:
from pyspark.sql import functions as F

def run_data_quality_checks():
    silver_df = spark.table("workspace.fraud_lakehouse.silver_transactions")
    gold_alerts_df = spark.table("workspace.fraud_lakehouse.gold_high_risk_alerts")

    checks = []

    # Check 1: Primary Key Uniqueness in Silver
    total_rows = silver_df.count()
    distinct_txns = silver_df.select("transaction_id").distinct().count()
    checks.append(("silver_pk_uniqueness", total_rows == distinct_txns, f"Total: {total_rows}, Distinct: {distinct_txns}"))

    # Check 2: No Negative or Null USD Amounts in Silver
    invalid_amounts = silver_df.filter((F.col("amount_usd").isNull()) | (F.col("amount_usd") <= 0)).count()
    checks.append(("silver_positive_amounts", invalid_amounts == 0, f"Invalid rows: {invalid_amounts}"))

    # Check 3: Risk Score Bounded Between 0 and 100
    out_of_bounds_scores = silver_df.filter((F.col("composite_risk_score") < 0) | (F.col("composite_risk_score") > 100)).count()
    checks.append(("silver_risk_score_bounds", out_of_bounds_scores == 0, f"Out of bounds: {out_of_bounds_scores}"))

    # Check 4: Gold Alerts Only Contain HIGH or CRITICAL Tiers
    invalid_alert_tiers = gold_alerts_df.filter(~F.col("risk_tier").isin("HIGH", "CRITICAL")).count()
    checks.append(("gold_alert_tier_contract", invalid_alert_tiers == 0, f"Non-high-risk rows leaked: {invalid_alert_tiers}"))

    # Evaluate and Fail Fast if any contract is violated
    failed_checks = [c for c in checks if not c[1]]
    for name, passed, details in checks:
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {name} -> {details}")

    if failed_checks:
        raise ValueError(f"Data Quality Contract Failed! Violations: {failed_checks}")

run_data_quality_checks()